# Step 2 — Indexing: Incidence Matrix dan Inverted Index

**Tujuan praktikum**
1. Mahasiswa memahami representasi *term-document incidence matrix* dan kelemahannya.
2. Mahasiswa mampu membangun *inverted index* (kamus + *posting list*) beserta *document frequency* dan *term frequency*.
3. Mahasiswa mampu membangun *positional index* dan menggunakannya untuk *phrase query*.

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from collections import Counter, defaultdict
from stki_utils import corpus_texts, preprocess

teks = corpus_texts()                      # {doc_id: judul + isi}
docs = {d: preprocess(t) for d, t in teks.items()}
print("Jumlah dokumen:", len(docs))
print("D07:", docs["D07"])

Jumlah dokumen: 30
D07: ['budidaya', 'padi', 'sawah', 'budidaya', 'padi', 'sawah', 'butuh', 'air', 'cukup', 'pupu', 'tepat', 'tani', 'anjur', 'guna', 'benih', 'unggul', 'hasil', 'panen', 'tingkat', 'hama', 'wereng', 'cokelat', 'jadi', 'ancam', 'utama', 'tanam', 'padi']


## Langkah 1 — Term-document incidence matrix
Matriks biner berukuran |V| × N: bernilai 1 jika term muncul di dokumen. Kita coba untuk 6 dokumen topik pertanian.

In [3]:
subset = ["D07", "D08", "D09", "D10", "D11", "D12"]
vocab_subset = sorted({t for d in subset for t in docs[d]})
incidence = pd.DataFrame([[int(t in docs[d]) for d in subset] for t in vocab_subset],
                         index=vocab_subset, columns=subset)
print("Ukuran matriks:", incidence.shape)
incidence.loc[["air", "hama", "padi", "pupuk", "sawah", "tani", "tanah"]]

Ukuran matriks: (109, 6)


,D07,D08,D09,D10,D11,D12
air,1,0,1,1,0,0
hama,1,0,0,0,1,0
padi,1,0,0,0,0,0
pupuk,0,1,0,1,0,0
sawah,1,0,0,0,1,0
tani,1,1,1,1,1,1
tanah,0,1,1,1,0,0


In [4]:
vocab = sorted({t for tokens in docs.values() for t in tokens})
sel_total = len(vocab) * len(docs)
sel_isi = sum(len(set(tokens)) for tokens in docs.values())
print(f"Kosakata: {len(vocab)} term, {len(docs)} dokumen")
print(f"Sel matriks penuh : {sel_total}")
print(f"Sel bernilai 1    : {sel_isi} ({sel_isi / sel_total:.1%})")

Kosakata: 472 term, 30 dokumen
Sel matriks penuh : 14160
Sel bernilai 1    : 702 (5.0%)


Sebagian besar sel bernilai 0 (*sparse*). Pada koleksi nyata berisi jutaan dokumen, matriks penuh tidak mungkin disimpan. Solusinya adalah **inverted index**: untuk setiap term hanya disimpan daftar dokumen yang memuatnya.

## Langkah 2 — Membangun inverted index
Struktur yang dibangun: `index[term] = {doc_id: tf}`. Panjang posting list sama dengan *document frequency* (df).

In [5]:
def build_inverted_index(docs_tokens):
    index = defaultdict(dict)
    for doc_id, tokens in docs_tokens.items():
        for term, tf in Counter(tokens).items():
            index[term][doc_id] = tf
    return dict(sorted(index.items()))

index = build_inverted_index(docs)
print("Jumlah term di kamus:", len(index))
for term in ["tani", "digital", "gizi", "cari"]:
    print(f"{term:8s} df={len(index[term]):2d}  postings={index[term]}")

Jumlah term di kamus: 472
tani     df= 7  postings={'D07': 1, 'D08': 2, 'D09': 1, 'D10': 3, 'D11': 1, 'D12': 3, 'D23': 1}
digital  df= 5  postings={'D19': 1, 'D21': 3, 'D25': 1, 'D27': 3, 'D29': 3}
gizi     df= 2  postings={'D02': 2, 'D30': 3}
cari     df= 2  postings={'D13': 3, 'D29': 1}


In [6]:
tabel = pd.DataFrame({
    "df": {t: len(p) for t, p in index.items()},
    "cf (total tf)": {t: sum(p.values()) for t, p in index.items()},
}).sort_values("df", ascending=False)
tabel.head(10)

,df,cf (total tf)
guna,11,12
lalu,9,9
bantu,8,8
tani,7,12
tinggi,6,7
tingkat,6,6
data,6,9
jadi,6,7
ajar,6,11
manfaat,5,5


## Langkah 3 — Menyimpan dan memuat index

In [7]:
import json, os

os.makedirs("output", exist_ok=True)
with open("output/inverted_index.json", "w") as f:
    json.dump(index, f)

with open("output/inverted_index.json") as f:
    index_dimuat = json.load(f)
print("Index tersimpan dan dimuat ulang dengan benar:", index_dimuat == index)
print("Ukuran file:", os.path.getsize("output/inverted_index.json"), "byte")

Index tersimpan dan dimuat ulang dengan benar: True
Ukuran file: 12606 byte


## Langkah 4 — Positional index dan phrase query
*Positional index* menyimpan posisi kemunculan term: `index[term][doc_id] = [posisi, ...]`. Dengan posisi, kita bisa mencari frasa, yaitu term yang muncul berurutan.

In [8]:
def build_positional_index(docs_tokens):
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in docs_tokens.items():
        for pos, term in enumerate(tokens):
            index[term][doc_id].append(pos)
    return {t: dict(p) for t, p in sorted(index.items())}

pos_index = build_positional_index(docs)
print("cerdas :", pos_index["cerdas"])
print("buat   :", pos_index["buat"])

cerdas : {'D10': [1, 5], 'D14': [0, 3], 'D27': [9]}
buat   : {'D08': [0, 5, 22], 'D14': [1, 4]}


In [9]:
def phrase_search(frasa, pos_index):
    terms = preprocess(frasa)
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["kecerdasan buatan", "pertanian cerdas", "pupuk organik", "data pribadi"]:
    print(f"{frasa!r:22s} -> {phrase_search(frasa, pos_index)}")

'kecerdasan buatan'    -> ['D14']
'pertanian cerdas'     -> ['D10']
'pupuk organik'        -> ['D08']
'data pribadi'         -> ['D15']


## Tugas Step 2
1. Hitung perbandingan ukuran (jumlah entri) antara incidence matrix penuh dan inverted index untuk seluruh korpus. Berapa kali lebih hemat inverted index?
2. Tampilkan 10 term dengan df = 1 (*hapax*). Apakah term-term tersebut berguna untuk pencarian? Jelaskan.
3. Posisi pada positional index di atas dihitung **setelah** stopword dihapus. Apa akibatnya pada pencarian frasa `"sistem temu kembali informasi"` atau `"gerakan 3M"`? Modifikasi kode agar posisi dihitung sebelum stopword dihapus.
4. Buat fungsi `proximity_search(t1, t2, k)` yang mengembalikan dokumen yang memuat t1 dan t2 dengan jarak paling jauh k posisi.

## 1. Incidence matrix vs inverted index

- *Incidence matrix*: menyimpan semua sel |V| × N (termasuk yang bernilai 0).
- *Inverted index*: hanya menyimpan pasangan (term, dokumen) yang benar-benar ada, yaitu jumlah seluruh *posting*.

In [10]:
sel_matriks   = len(vocab) * len(docs)                       # semua sel, termasuk 0
total_posting = sum(len(p) for p in index.values())        # jumlah entri posting di inverted index
entri_dengan_kamus = total_posting + len(index)            # posting + 1 entri kamus untuk tiap term

print(f"Incidence matrix penuh        : {len(vocab)} x {len(docs)} = {sel_matriks} sel")
print(f"Inverted index (posting saja) : {total_posting} entri")
print(f"Inverted index (+ kamus term) : {entri_dengan_kamus} entri")
print()
print(f"Lebih hemat (posting saja)    : {sel_matriks / total_posting:.1f} kali")
print(f"Lebih hemat (+ kamus term)    : {sel_matriks / entri_dengan_kamus:.1f} kali")

Incidence matrix penuh        : 472 x 30 = 14160 sel
Inverted index (posting saja) : 702 entri
Inverted index (+ kamus term) : 1174 entri

Lebih hemat (posting saja)    : 20.2 kali
Lebih hemat (+ kamus term)    : 12.1 kali


*Kesimpulan:* inverted index sekitar **20,2× lebih hemat** (14.160 sel → 702 posting), atau sekitar **12,1×** jika 472 entri kamus term ikut dihitung. Hemat karena hanya ≈5% sel matriks yang bernilai 1; sisanya 0 dan tidak perlu disimpan. Pada koleksi besar selisihnya jauh lebih besar karena kosakata dan jumlah dokumen bertambah, sementara jumlah kata per dokumen relatif tetap.

## 2. Term dengan df = 1 (hapax legomena)

*Hapax* adalah term yang hanya muncul di *satu* dokumen.

In [11]:
hapax = [t for t, p in index.items() if len(p) == 1]
print(f"Jumlah hapax: {len(hapax)} dari {len(index)} term ({len(hapax)/len(index):.0%})")

contoh_hapax = sorted(hapax)[:10]

df_hapax = pd.DataFrame([{"term": t, "df": len(index[t]), "dokumen": list(index[t])[0]} for t in contoh_hapax])
df_hapax
pd.DataFrame([{"term": t, "df": len(index[t]), "dokumen": list(index[t])[0]} for t in contoh_hapax])

Jumlah hapax: 346 dari 472 term (73%)


,term,df,dokumen
0,150,1,D03
1,2,1,D04
2,3m,1,D01
3,acu,1,D20
4,aedes,1,D01
5,aegypti,1,D01
6,ahli,1,D03
7,aju,1,D22
8,akibat,1,D02
9,aktif,1,D15


In [12]:
# 10 hapax pertama menurut urutan alfabet (sebagai pembanding)
hapax[:10]

['150', '2', '3m', 'acu', 'aedes', 'aegypti', 'ahli', 'aju', 'akibat', 'aktif']

*Apakah berguna untuk pencarian? tergantung jenis term-nya.*

- *Berguna* untuk term spesifik/langka seperti `aedes`, `aegypti`, `arabika`, `autentikasi`, `beasiswa`. Karena hanya muncul di satu dokumen, term ini punya daya pembeda (IDF) sangat tinggi sehingga query yang memuatnya langsung menunjuk dokumen yang tepat.
- *Kurang/tidak berguna* untuk term generik atau angka seperti `atas`, `banyak`, `beberapa`, `150`, `2`. Kata-kata ini kebetulan muncul sekali saja, bukan karena penting. Mereka hanya menambah ukuran kamus tanpa membantu pencarian (kandidat stopword tambahan).
- Pada korpus kecil ini *jumlah hapax yang tercetak pada output kode di atas adalah hapax*, sesuai hukum Zipf: sebagian besar term jarang muncul. Karena itu hapax tidak boleh dibuang begitu saja; ia perlu dipilah antara term informatif (nama/istilah khusus) dan *noise*.

## 3. Posisi dihitung *sebelum* stopword dihapus

*Masalah.* Pada kode awal, stopword dibuang dulu baru posisi dihitung. Akibatnya kata yang aslinya terpisah oleh stopword menjadi *tampak bersebelahan*. Pada D29 teksnya adalah:

> "Sistem temu *kembali* informasi ..." — `kembali` termasuk stopword Sastrawi.

In [13]:
print("Preprocess dengan stopword removal :", preprocess("sistem temu kembali informasi"))
print("Preprocess tanpa  stopword removal :", preprocess("sistem temu kembali informasi", use_stopword=False))
print()
# Dengan index lama, frasa palsu "sistem temu informasi" (tanpa 'kembali') ikut cocok:
for frasa in ["sistem temu kembali informasi", "sistem temu informasi", "gerakan 3M"]:
    print(f"LAMA  {frasa!r:34s} -> {phrase_search(frasa, pos_index)}")

Preprocess dengan stopword removal : ['sistem', 'temu', 'informasi']
Preprocess tanpa  stopword removal : ['sistem', 'temu', 'kembali', 'informasi']

LAMA  'sistem temu kembali informasi'    -> ['D29']
LAMA  'sistem temu informasi'            -> ['D29']
LAMA  'gerakan 3M'                       -> ['D01']


Frasa `"sistem temu informasi"` berpotensi menjadi *false positive* karena `kembali` dihapus dan `temu` dan `informasi` tampak berdampingan. Hasil aktual dapat dilihat dari pengujian pada kedua indeks.

*Perbaikan:* hitung posisi dari token yang *masih lengkap (stopword dipertahankan)*, lalu gunakan perlakuan yang sama pada query.

In [14]:
# Docs dengan stopword dipertahankan (hanya stemming) -> posisi mencerminkan teks asli
docs_penuh = {d: preprocess(t, use_stopword=False) for d, t in teks.items()}
pos_index_penuh = build_positional_index(docs_penuh)

def phrase_search_v2(frasa, pos_index):
    terms = preprocess(frasa, use_stopword=False)      # <- query juga tanpa stopword removal
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["sistem temu kembali informasi", "sistem temu informasi", "gerakan 3M",
              "kecerdasan buatan", "pertanian cerdas", "pupuk organik", "data pribadi"]:
    print(f"{frasa!r:34s} LAMA: {str(phrase_search(frasa, pos_index)):8s} BARU: {phrase_search_v2(frasa, pos_index_penuh)}")

'sistem temu kembali informasi'    LAMA: ['D29']  BARU: ['D29']
'sistem temu informasi'            LAMA: ['D29']  BARU: []
'gerakan 3M'                       LAMA: ['D01']  BARU: ['D01']
'kecerdasan buatan'                LAMA: ['D14']  BARU: ['D14']
'pertanian cerdas'                 LAMA: ['D10']  BARU: ['D10']
'pupuk organik'                    LAMA: ['D08']  BARU: ['D08']
'data pribadi'                     LAMA: ['D15']  BARU: ['D15']


*Hasil dan akibatnya*

- Frasa `"sistem temu kembali informasi"` memang terdapat pada **D29** dan semestinya ditemukan oleh index yang mempertahankan posisi asli.
- Frasa `"sistem temu informasi"` tidak berurutan persis pada dokumen asli, sehingga seharusnya **tidak cocok**. Jika stopword dihapus lalu posisi dirapatkan, muncul risiko *false positive*.
- `"gerakan 3M"` terdapat pada **D01**, dan keduanya tidak dipisahkan stopword sehingga hasil ideal tetap sama.

*Trade-off:* menyimpan stopword membuat indeks lebih besar, tetapi menjaga kecocokan frasa. Alternatifnya, simpan hanya posting term penting namun pertahankan offset dari token sebelum stopword dihapus.

## 4. Fungsi `proximity_search(t1, t2, k)`

Mengembalikan dokumen yang memuat `t1` dan `t2` dengan *jarak paling jauh k posisi*.

*Langkah:* (1) preprocess kedua term, (2) ambil dokumen yang memuat keduanya (irisan), (3) di tiap dokumen cek apakah ada pasangan posisi `p1`, `p2` dengan `|p1 − p2| ≤ k`.

In [15]:
def proximity_search(t1, t2, k, pos_index):
    a, b = preprocess(t1, use_stopword=False), preprocess(t2, use_stopword=False)
    if len(a) != 1 or len(b) != 1:
        raise ValueError("t1 dan t2 masing-masing harus satu kata")
    a, b = a[0], b[0]
    if a not in pos_index or b not in pos_index:
        return []

    hasil = []
    for d in sorted(set(pos_index[a]) & set(pos_index[b])):       # dokumen yang memuat keduanya
        if any(abs(p - q) <= k
               for p in pos_index[a][d] for q in pos_index[b][d]
               if not (a == b and p == q)):                        # jangan bandingkan posisi dengan dirinya sendiri
            hasil.append(d)
    return hasil

for t1, t2, k in [("sistem", "informasi", 1), ("sistem", "informasi", 3), ("sistem", "informasi", 5),
                  ("pertanian", "cerdas", 2), ("pupuk", "organik", 1), ("data", "pribadi", 3), ("air", "hama", 10)]:
    print(f"proximity_search({t1!r}, {t2!r}, k={k}) -> {proximity_search(t1, t2, k, pos_index_penuh)}")

proximity_search('sistem', 'informasi', k=1) -> []
proximity_search('sistem', 'informasi', k=3) -> ['D29']
proximity_search('sistem', 'informasi', k=5) -> ['D29']
proximity_search('pertanian', 'cerdas', k=2) -> ['D10']
proximity_search('pupuk', 'organik', k=1) -> ['D08']
proximity_search('data', 'pribadi', k=3) -> ['D15']
proximity_search('air', 'hama', k=10) -> []


*Cara membaca hasil:* pemeriksaan posisi `sistem` dan `informasi` pada D29 memperlihatkan bahwa *proximity search* dapat menemukan kedua term meskipun tidak berdampingan. Frasa merupakan bentuk lebih ketat, karena kata harus berada pada urutan dan posisi persis. Perhatikan hasil jarak k pada sel kode sebelumnya untuk mengetahui jarak minimal aktual.